# COLLABORATIVE FILTERING ENGINE (SVD)

## 1. Import necessary libaries

In [2]:
import pandas as pd
import numpy as np
import os
import pickle
from tqdm import tqdm
import warnings
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split, cross_validate
from surprise import accuracy
warnings.filterwarnings('ignore')

## 2. Load Processed Data 

In [3]:
ratings_path = '../data/ml-latest-small/ratings.csv'

if not os.path.exists(ratings_path):
    raise FileNotFoundError(f"Ratings file not found: {ratings_path}")

ratings = pd.read_csv(ratings_path)
print(f"   Loaded {len(ratings)} ratings")
print(f"   Unique users : {ratings['userId'].nunique()}")
print(f"   Unique movies: {ratings['movieId'].nunique()}")
print("\n   Rating distribution:")
print(ratings['rating'].value_counts().sort_index())

   Loaded 100836 ratings
   Unique users : 610
   Unique movies: 9724

   Rating distribution:
rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64


## 3. Data Cleaning

In [4]:
ratings.head(3)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224


### 3a. Checking Null Values

In [6]:
ratings.shape

(100836, 4)

In [8]:
ratings.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

# 4. Model Training

In [9]:
# Prepare data for suprise

# Surprise requires a specific format: userId, itemId (movieId), rating
# We use Reader to define the rating scale (0.5 to 5.0 in MovieLens)
reader = Reader(rating_scale=(0.5, 5.0))

# Load the dataframe into Surprise Dataset
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

In [10]:
# Train-Test Split and Cross Validation.

# Use SVD algorithm (matrix factorization)
# Number of latent factors (n_facto) - good balance for small dataset
algo = SVD(n_factors=50,  
           n_epochs=20, 
           lr_all=0.005, 
           reg_all=0.02,
           random_state=42)

# Run 5-fold cross validation (fast on ml-latest-small)
cv_results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

print(f"\n   Average RMSE: {cv_results['test_rmse'].mean():.4f}")
print(f"   Average MAE : {cv_results['test_mae'].mean():.4f}")

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8733  0.8625  0.8687  0.8679  0.8794  0.8703  0.0057  
MAE (testset)     0.6710  0.6603  0.6680  0.6684  0.6755  0.6686  0.0049  
Fit time          0.30    0.30    0.35    0.30    0.30    0.31    0.02    
Test time         0.04    0.09    0.04    0.04    0.08    0.06    0.02    

   Average RMSE: 0.8703
   Average MAE : 0.6686


In [11]:
# Train final model on full dataset.

# Build full trainset (no test split for final model)
trainset = data.build_full_trainset()

# Train the model
algo.fit(trainset)

In [12]:
# Saving the trained SVD model.

models_dir = '../artifacts'
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, 'svd_model.pkl')

with open(model_path, 'wb') as f:
    pickle.dump(algo, f)

print(f"Model saved to: {model_path}")

# Also save some useful metadata
model_metadata = {
    'algorithm': 'SVD',
    'n_factors': 50,
    'n_epochs': 20,
    'rmse_cv': float(cv_results['test_rmse'].mean()),
    'mae_cv': float(cv_results['test_mae'].mean()),
    'num_users': ratings['userId'].nunique(),
    'num_movies': ratings['movieId'].nunique()
}

metadata_path = os.path.join(models_dir, 'svd_metadata.pkl')
with open(metadata_path, 'wb') as f:
    pickle.dump(model_metadata, f)

print(f"Metadata saved to: {metadata_path}")

Model saved to: ../artifacts/svd_model.pkl
Metadata saved to: ../artifacts/svd_metadata.pkl


## 6. Test Similarity Function

In [14]:


def get_collaborative_recommendations(user_id, top_n=10):
    """Get top-N movie recommendations for a specific user using SVD."""
    
    # Get all movieIds the user has NOT rated yet
    user_rated = ratings[ratings['userId'] == user_id]['movieId'].unique()
    
    # Load enriched movies to map movieId → title
    enriched_path = '../data/processed/enriched_movies.csv'
    enriched_movies = pd.read_csv(enriched_path)
    
    # All possible movies
    all_movie_ids = enriched_movies['movieId'].unique()
    unrated_movies = [mid for mid in all_movie_ids if mid not in user_rated]
    
    # Predict ratings for all unrated movies
    predictions = []
    for movie_id in unrated_movies:
        pred = algo.predict(uid=user_id, iid=movie_id)
        predictions.append((movie_id, pred.est))
    
    # Sort by predicted rating (descending)
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    # Get top N
    top_predictions = predictions[:top_n]
    
    # Map to titles and return nice DataFrame
    results = []
    for movie_id, est_rating in top_predictions:
        title = enriched_movies[enriched_movies['movieId'] == movie_id]['title'].iloc[0]
        results.append({
            'movieId': movie_id,
            'title': title,
            'predicted_rating': round(est_rating, 2)
        })
    
    return pd.DataFrame(results)

In [15]:

try:
    test_rec = get_collaborative_recommendations(user_id=1, top_n=8)
    print(test_rec[['title', 'predicted_rating']])
except Exception as e:
    print(f"   Test failed: {e} (maybe user 1 has rated too many movies)")

print("📁 Files created in ../artifacts/:")
print("   • svd_model.pkl")
print("   • svd_metadata.pkl")
print(f"\nModel Performance (5-fold CV):")
print(f"   RMSE: {cv_results['test_rmse'].mean():.4f} | MAE: {cv_results['test_mae'].mean():.4f}")

                                               title  predicted_rating
0                   Shawshank Redemption, The (1994)               5.0
1  Dr. Strangelove or: How I Learned to Stop Worr...               5.0
2                              Godfather, The (1972)               5.0
3                         Singin' in the Rain (1952)               5.0
4                                 Rear Window (1954)               5.0
5                                  Casablanca (1942)               5.0
6                   Streetcar Named Desire, A (1951)               5.0
7  Good, the Bad and the Ugly, The (Buono, il bru...               5.0
📁 Files created in ../artifacts/:
   • svd_model.pkl
   • svd_metadata.pkl

Model Performance (5-fold CV):
   RMSE: 0.8703 | MAE: 0.6686
